# East River ASO spatial metrics — SWE, snow depth, and bulk snow density

This notebook uses the **existing East River Workflow v0.8.2 common-Noah-grid outputs directly**.
There is **no `aso_spatial_metrics.csv` input** in this version.

It calculates the ASO/model spatial metrics separately for the workflow's four
elevation bands on the **common Noah-MP grid**:

- **Lower:** elevation < 2896 m
- **Middle:** 2896 m <= elevation < 3353 m
- **Upper:** elevation >= 3353 m
- **All:** all valid East River watershed cells

For SWE and snow depth, the notebook reports:

- **Pearson `r`**
- **MB (mm)**
- **RMB (%)**
- **RMSE (mm)**
- **RRMSE (%)**
- paired-cell sample size `n`

The relative metrics are

\[
\mathrm{RMB}(\%) = 100\,\frac{\mathrm{MB}}{\overline{\mathrm{ASO}}}
\]

\[
\mathrm{RRMSE}(\%) = 100\,\frac{\mathrm{RMSE}}{\overline{\mathrm{ASO}}}
\]

For each model/band, \(\overline{\mathrm{ASO}}\) is computed on the same finite
paired common-grid support used for that model's error distribution. RMB preserves
the workflow bias sign convention: **model - ASO**.

## Bulk snow-density extension

Bulk snow density is calculated **after** SWE and snow depth have been placed on
the common Noah-MP grid:

\[
\rho = 1000\,\frac{\mathrm{SWE\ (mm)}}{\mathrm{snow\ depth\ (mm)}}
\qquad [\mathrm{kg\,m^{-3}}].
\]

The density comparison uses the agreed reference-based screening rule:

1. use the common Noah-MP grid;
2. retain cells where **ASO snow depth > 100 mm**;
3. require finite ASO, iSnobal, and Noah-MP SWE and snow depth on the same cells;
4. require positive iSnobal and Noah-MP snow depth so density is mathematically defined;
5. calculate ASO, iSnobal, and Noah-MP bulk density cell by cell;
6. use the **same common cell support** for both model comparisons;
7. treat each retained common-grid cell as one equally weighted spatial observation;
8. calculate ASO mean density, Pearson `r`, MB, RMB, RMSE, and RRMSE.

No upper-density clipping is applied. The density analysis inherits the ASO version
already used to create the workflow's `regridded/` products; this notebook itself
does **not** run additional ASO quality control.

## Scientific lineage

- Elevation limits come directly from the workflow `ELEVATION_BANDS` constants.
- Elevation masks use the workflow's `elevation_band_mask()` function and the
  native Noah-MP DEM used by Table 6.
- SWE/snow-depth **MB and RMSE** are taken from the validated
  `tables/table6_aso_validation.csv`.
- The notebook independently recomputes SWE/snow-depth MB/RMSE from the saved
  difference rasters within every elevation band and stops if they conflict with
  Table 6.
- Pearson `r` is calculated from paired ASO/model values on the same common
  Noah-MP grid.
- Bulk-density metrics are calculated directly from the saved common-grid SWE and
  snow-depth rasters using the explicit ASO snow-depth threshold above.


In [ ]:

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import rasterio

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.lines import Line2D
from matplotlib.transforms import blended_transform_factory


In [ ]:
# ============================================================
# PROJECT PATHS
# ============================================================

# User-confirmed East River Workflow v0.8.2 output directory.
WORKFLOW_OUTPUT_DIR = Path(
    "/bsuhome/zihaozhang/scratch/iSnobal/"
    "East_River_Workflow_Code_v0.8.2/output/output1"
)

# Repository root is two levels above output/output1.
REPO_ROOT = WORKFLOW_OUTPUT_DIR.parents[1]
SRC_DIR = REPO_ROOT / "src"
CONFIG_PATH = REPO_ROOT / "config" / "east_river_config.yaml"

# Author-selected unfiltered ASO; regenerate from originals, never cached ASO.
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from prepare_unfiltered_aso import prepare
ASO_PUBLICATION_DIR = prepare(CONFIG_PATH, WORKFLOW_OUTPUT_DIR, native=False)

TABLE_OUTPUT_DIR = ASO_PUBLICATION_DIR / "tables"
TABLE6_PATH = TABLE_OUTPUT_DIR / "table6_aso_validation.csv"
WEIGHT_PATH = WORKFLOW_OUTPUT_DIR / "cache" / "east_river_fraction_noahmp.npz"
REGRID_DIR = ASO_PUBLICATION_DIR / "regridded"
FIGURE_OUTPUT_DIR = ASO_PUBLICATION_DIR / "figures" / "scientific"

required = {
    "workflow output": WORKFLOW_OUTPUT_DIR,
    "repository src": SRC_DIR / "east_river_workflow",
    "workflow config": CONFIG_PATH,
    "Table 6": TABLE6_PATH,
    "Noah watershed weights": WEIGHT_PATH,
    "regridded raster directory": REGRID_DIR,
}

missing = [f"{name}: {path}" for name, path in required.items() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Required East River v0.8.2 input(s) are missing:\n  - "
        + "\n  - ".join(missing)
    )

TABLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Workflow output:", WORKFLOW_OUTPUT_DIR)
print("Repository root:", REPO_ROOT)
print("Config:", CONFIG_PATH)
print("Table 6:", TABLE6_PATH)
print("Regridded rasters:", REGRID_DIR)
print("Table output:", TABLE_OUTPUT_DIR)
print("Figure output:", FIGURE_OUTPUT_DIR)


In [ ]:
# ============================================================
# IMPORT VALIDATED WORKFLOW DEFINITIONS
# ============================================================

from east_river_workflow.constants import MODEL_COLORS, ELEVATION_BANDS
from east_river_workflow.utils import weighted_nanmean
from east_river_workflow.metrics import pearson_mean_bias
from east_river_workflow.aso import distribution_statistics, elevation_band_mask
from east_river_workflow.config import load_config
from east_river_workflow.data_access import read_noah_grid

# ASO/observations use black source identity.
ASO_COLOR = "black"

print("Workflow model colors:", MODEL_COLORS)
print("Workflow elevation bands:", ELEVATION_BANDS)


In [ ]:

# ============================================================
# EAST RIVER PUBLICATION PLOTTING PROTOCOL
# ============================================================

FONT_NAME = "Nimbus Sans"
BASE_FONT_SIZE = 18
PANEL_TITLE_SIZE = 20
MODEL_LINEWIDTH = 2.5
ASO_LINEWIDTH = 1.8
MARKER_SIZE = 7.5
SAVE_DPI = 600
FIGSIZE = (14.0, 11.5)
WY_GAP = 0.75

# GeoTIFFs are float32; use a small tolerance when checking against
# the in-memory statistics saved to Table 6.
TABLE_RASTER_TOLERANCE_MM = 1.0e-2

available_names = {f.name for f in fm.fontManager.ttflist}
if FONT_NAME not in available_names:
    raise RuntimeError(
        f"{FONT_NAME!r} is required by the current East River plotting protocol, "
        "but it is not available in this Python environment. "
        "No silent font fallback is allowed."
    )

mpl.rcParams.update({
    "font.family": FONT_NAME,
    "font.size": BASE_FONT_SIZE,
    "axes.titlesize": PANEL_TITLE_SIZE,
    "axes.titleweight": "normal",
    "axes.labelweight": "normal",
    "axes.linewidth": 1.0,
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
    "xtick.major.size": 5.0,
    "ytick.major.size": 5.0,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

print("Verified plotting font:", FONT_NAME)
print("iSnobal:", MODEL_COLORS["iSnobal"])
print("Noah-MP:", MODEL_COLORS["Noah-MP"])
print("ASO:", ASO_COLOR)


In [ ]:
# ============================================================
# LOAD TABLE 6, EXACT NOAH-GRID WEIGHTS, AND NOAH ELEVATION
# ============================================================

table6 = pd.read_csv(TABLE6_PATH)
table6["date"] = pd.to_datetime(table6["date"]).dt.normalize()

required_columns = {
    "date",
    "variable",
    "product",
    "elevation_band",
    "cell_count",
    "mean_mm",
    "rmse_mm",
}
missing_columns = required_columns - set(table6.columns)
if missing_columns:
    raise ValueError(
        f"Table 6 is missing required columns: {sorted(missing_columns)}"
    )

# Publication Table 6 should contain the two common-Noah-grid products used here.
PRODUCT_I = "iSnobal-ASO on Noah-MP grid"
PRODUCT_N = "Noah-MP-ASO on Noah-MP grid"

table6_common = table6[
    table6["product"].isin([PRODUCT_I, PRODUCT_N])
].copy()

expected_bands = list(ELEVATION_BANDS)
available_bands = set(table6_common["elevation_band"].dropna().astype(str))
missing_bands = [band for band in expected_bands if band not in available_bands]
if missing_bands:
    raise ValueError(
        "Table 6 is missing expected workflow elevation band(s): "
        + ", ".join(missing_bands)
    )

with np.load(WEIGHT_PATH, allow_pickle=False) as z:
    if "weights" not in z.files:
        raise KeyError(
            f"{WEIGHT_PATH} does not contain the expected 'weights' array. "
            f"Available keys: {z.files}"
        )
    noah_weights = np.asarray(z["weights"], dtype=float)

# Load the exact Noah-MP DEM used by the workflow for elevation-band summaries.
cfg = load_config(CONFIG_PATH)
cfg_output_dir = Path(cfg.output_dir).expanduser().resolve()
expected_output_dir = WORKFLOW_OUTPUT_DIR.expanduser().resolve()
if cfg_output_dir != expected_output_dir:
    raise ValueError(
        "Config/output mismatch: the workflow config points to "
        f"{cfg_output_dir}, but this notebook is reading {expected_output_dir}. "
        "Use the config associated with this output directory."
    )

_noah_grid, noah_dem, _noah_lat, _noah_lon, _noah_grid_diag = read_noah_grid(
    cfg.data["paths"]["noahmp_geo_file"],
    cfg.section("noahmp"),
)
noah_dem = np.asarray(noah_dem, dtype=float)

if noah_dem.shape != noah_weights.shape:
    raise ValueError(
        "Noah DEM/weight shape mismatch: "
        f"DEM={noah_dem.shape}, weights={noah_weights.shape}"
    )

print("Common-grid Table 6 rows:", len(table6_common))
print("Noah weight/DEM shape:", noah_weights.shape)
print("Variables:", sorted(table6_common["variable"].unique().tolist()))
print("Elevation bands:", ELEVATION_BANDS)


In [ ]:
# ============================================================
# RASTER ACCESS + ELEVATION-BAND METRICS + CONSISTENCY CHECKS
# ============================================================

def variable_slug(variable):
    return str(variable).lower().replace(" ", "_")


def raster_path(date, variable, key):
    date_text = pd.Timestamp(date).strftime("%Y%m%d")
    return REGRID_DIR / f"{date_text}_{variable_slug(variable)}_{key}.tif"


def read_raster(date, variable, key):
    path = raster_path(date, variable, key)
    if not path.exists():
        raise FileNotFoundError(f"Required workflow raster is missing: {path}")
    with rasterio.open(path) as src:
        arr = src.read(1, masked=True).filled(np.nan).astype(float)
    if arr.shape != noah_weights.shape:
        raise ValueError(
            f"Raster/weight shape mismatch for {path.name}: "
            f"raster={arr.shape}, weights={noah_weights.shape}"
        )
    return arr


def table_row(date, variable, product, elevation_band):
    row = table6_common[
        table6_common["date"].eq(pd.Timestamp(date).normalize())
        & table6_common["variable"].eq(variable)
        & table6_common["product"].eq(product)
        & table6_common["elevation_band"].eq(elevation_band)
    ]
    if len(row) != 1:
        raise ValueError(
            "Expected exactly one Table 6 row for "
            f"{pd.Timestamp(date).date()} / {variable} / {product} / "
            f"{elevation_band}; found {len(row)}."
        )
    return row.iloc[0]


def assert_metric_match(
    label,
    table_value,
    raster_value,
    date,
    variable,
    elevation_band,
):
    if np.isfinite(table_value) and np.isfinite(raster_value):
        if not np.isclose(
            float(table_value),
            float(raster_value),
            rtol=0.0,
            atol=TABLE_RASTER_TOLERANCE_MM,
        ):
            raise ValueError(
                f"{label} conflict for {pd.Timestamp(date).date()} "
                f"{variable} / {elevation_band}: "
                f"Table 6 = {table_value:.6f}, "
                f"saved-raster calculation = {raster_value:.6f}. "
                "Resolve this data-lineage conflict before using the metrics."
            )
    elif np.isfinite(table_value) != np.isfinite(raster_value):
        raise ValueError(
            f"{label} finite/NaN conflict for {pd.Timestamp(date).date()} "
            f"{variable} / {elevation_band}: "
            f"Table 6 = {table_value}, saved-raster calculation = {raster_value}."
        )


def assert_cell_count_match(
    label,
    table_count,
    raster_count,
    date,
    variable,
    elevation_band,
):
    table_count = int(table_count)
    raster_count = int(raster_count)
    if table_count != raster_count:
        raise ValueError(
            f"{label} cell-count conflict for {pd.Timestamp(date).date()} "
            f"{variable} / {elevation_band}: "
            f"Table 6 = {table_count}, saved-raster calculation = {raster_count}."
        )


def relative_metric_percent(metric_value, reference_mean):
    """Normalize an absolute metric by the mean ASO value on identical support."""
    metric_value = float(metric_value)
    reference_mean = float(reference_mean)

    if not np.isfinite(metric_value) or not np.isfinite(reference_mean):
        return np.nan
    if reference_mean <= 0.0:
        return np.nan

    return 100.0 * metric_value / reference_mean


def elevation_range_text(band):
    lower, upper = ELEVATION_BANDS[band]
    if lower is None and upper is None:
        return "All"
    if lower is None:
        return f"< {upper:g} m"
    if upper is None:
        return f">= {lower:g} m"
    return f"{lower:g} to < {upper:g} m"


def build_elevation_metrics():
    pairs = (
        table6_common[["date", "variable"]]
        .drop_duplicates()
        .sort_values(["variable", "date"])
    )

    rows = []
    watershed_mask = (
        np.isfinite(noah_weights)
        & (noah_weights > 0)
        & np.isfinite(noah_dem)
    )

    for item in pairs.itertuples(index=False):
        date = pd.Timestamp(item.date)
        variable = str(item.variable)

        aso = read_raster(date, variable, "aso_on_noah")
        isnobal = read_raster(date, variable, "isnobal_on_noah")
        noah = read_raster(date, variable, "noah_native")

        diff_i = read_raster(date, variable, "difference_isnobal_noah")
        diff_n = read_raster(date, variable, "difference_noah")

        water_year = date.year + 1 if date.month >= 10 else date.year

        for elevation_band in ELEVATION_BANDS:
            lower_m, upper_m = ELEVATION_BANDS[elevation_band]
            band_mask = (
                watershed_mask
                & elevation_band_mask(noah_dem, elevation_band)
            )

            # Model-independent, fractional-area-weighted ASO magnitude for the
            # band. This is descriptive only; RMB/RRMSE use model-specific paired
            # ASO support below so the denominator exactly matches each error
            # distribution.
            band_weights = np.where(band_mask, noah_weights, 0.0)
            aso_band_mean_mm, aso_band_valid_area_fraction = weighted_nanmean(
                aso,
                band_weights,
            )

            # Recompute the workflow's unweighted difference distributions on the
            # exact Table-6 elevation support.
            stats_i = distribution_statistics(diff_i[band_mask])
            stats_n = distribution_statistics(diff_n[band_mask])

            row_i = table_row(date, variable, PRODUCT_I, elevation_band)
            row_n = table_row(date, variable, PRODUCT_N, elevation_band)

            mb_i = float(row_i["mean_mm"])
            mb_n = float(row_n["mean_mm"])
            rmse_i = float(row_i["rmse_mm"])
            rmse_n = float(row_n["rmse_mm"])

            assert_cell_count_match(
                "iSnobal",
                row_i["cell_count"],
                stats_i["cell_count"],
                date,
                variable,
                elevation_band,
            )
            assert_cell_count_match(
                "Noah-MP",
                row_n["cell_count"],
                stats_n["cell_count"],
                date,
                variable,
                elevation_band,
            )
            assert_metric_match(
                "iSnobal mean bias",
                mb_i,
                stats_i["mean_mm"],
                date,
                variable,
                elevation_band,
            )
            assert_metric_match(
                "Noah-MP mean bias",
                mb_n,
                stats_n["mean_mm"],
                date,
                variable,
                elevation_band,
            )
            assert_metric_match(
                "iSnobal RMSE",
                rmse_i,
                stats_i["rmse_mm"],
                date,
                variable,
                elevation_band,
            )
            assert_metric_match(
                "Noah-MP RMSE",
                rmse_n,
                stats_n["rmse_mm"],
                date,
                variable,
                elevation_band,
            )

            # Exact finite paired supports for Pearson r and the relative-metric
            # denominators. Supports are model-specific because one model can have
            # a missing cell where the other does not.
            paired_mask_i = (
                band_mask
                & np.isfinite(aso)
                & np.isfinite(isnobal)
                & np.isfinite(diff_i)
            )
            paired_mask_n = (
                band_mask
                & np.isfinite(aso)
                & np.isfinite(noah)
                & np.isfinite(diff_n)
            )

            r_i_stats = pearson_mean_bias(
                aso[paired_mask_i],
                isnobal[paired_mask_i],
            )
            r_n_stats = pearson_mean_bias(
                aso[paired_mask_n],
                noah[paired_mask_n],
            )

            # The paired values used for r should reproduce Table 6 mean bias and
            # cell count on the same support.
            assert_metric_match(
                "iSnobal paired-value mean bias",
                mb_i,
                r_i_stats["mean_bias"],
                date,
                variable,
                elevation_band,
            )
            assert_metric_match(
                "Noah-MP paired-value mean bias",
                mb_n,
                r_n_stats["mean_bias"],
                date,
                variable,
                elevation_band,
            )
            assert_cell_count_match(
                "iSnobal paired support",
                row_i["cell_count"],
                r_i_stats["n"],
                date,
                variable,
                elevation_band,
            )
            assert_cell_count_match(
                "Noah-MP paired support",
                row_n["cell_count"],
                r_n_stats["n"],
                date,
                variable,
                elevation_band,
            )

            aso_reference_mean_i_mm = (
                float(np.mean(aso[paired_mask_i]))
                if np.any(paired_mask_i)
                else np.nan
            )
            aso_reference_mean_n_mm = (
                float(np.mean(aso[paired_mask_n]))
                if np.any(paired_mask_n)
                else np.nan
            )

            rows.append({
                "date": date,
                "water_year": int(water_year),
                "variable": variable,
                "elevation_band": elevation_band,
                "elevation_range": elevation_range_text(elevation_band),
                "elevation_lower_m": (
                    np.nan if lower_m is None else float(lower_m)
                ),
                "elevation_upper_m": (
                    np.nan if upper_m is None else float(upper_m)
                ),
                "aso_band_weighted_mean_mm": float(aso_band_mean_mm),
                "aso_band_valid_area_fraction": float(
                    aso_band_valid_area_fraction
                ),
                "aso_reference_mean_isnobal_support_mm": (
                    aso_reference_mean_i_mm
                ),
                "aso_reference_mean_noahmp_support_mm": (
                    aso_reference_mean_n_mm
                ),
                "mb_isnobal_mm": mb_i,
                "rmb_isnobal_pct": relative_metric_percent(
                    mb_i,
                    aso_reference_mean_i_mm,
                ),
                "rmse_isnobal_mm": rmse_i,
                "rrmse_isnobal_pct": relative_metric_percent(
                    rmse_i,
                    aso_reference_mean_i_mm,
                ),
                "r_isnobal": float(r_i_stats["pearson_r"]),
                "n_isnobal": int(r_i_stats["n"]),
                "mb_noahmp_mm": mb_n,
                "rmb_noahmp_pct": relative_metric_percent(
                    mb_n,
                    aso_reference_mean_n_mm,
                ),
                "rmse_noahmp_mm": rmse_n,
                "rrmse_noahmp_pct": relative_metric_percent(
                    rmse_n,
                    aso_reference_mean_n_mm,
                ),
                "r_noahmp": float(r_n_stats["pearson_r"]),
                "n_noahmp": int(r_n_stats["n"]),
            })

    band_order = {band: i for i, band in enumerate(ELEVATION_BANDS)}
    out = pd.DataFrame(rows)
    out["_band_order"] = out["elevation_band"].map(band_order)
    out = (
        out.sort_values(["variable", "date", "_band_order"])
        .drop(columns="_band_order")
        .reset_index(drop=True)
    )
    return out


metrics = build_elevation_metrics()
metrics_all = metrics[metrics["elevation_band"].eq("All")].copy()

metric_review_columns = [
    "date",
    "variable",
    "elevation_band",
    "elevation_range",
    "aso_reference_mean_isnobal_support_mm",
    "mb_isnobal_mm",
    "rmb_isnobal_pct",
    "rmse_isnobal_mm",
    "rrmse_isnobal_pct",
    "r_isnobal",
    "n_isnobal",
    "aso_reference_mean_noahmp_support_mm",
    "mb_noahmp_mm",
    "rmb_noahmp_pct",
    "rmse_noahmp_mm",
    "rrmse_noahmp_pct",
    "r_noahmp",
    "n_noahmp",
]

metrics[metric_review_columns]


In [ ]:
# ============================================================
# WRITE METRICS TABLES
# ============================================================

# Main requested table: every date x variable x elevation band.
elevation_metrics_path = (
    TABLE_OUTPUT_DIR / "aso_spatial_metrics_by_elevation.csv"
)
metrics.to_csv(elevation_metrics_path, index=False)

# Figure audit table: retain only the All band because the existing 3x1 figures
# are whole-watershed figures.
figure_audit_path = (
    FIGURE_OUTPUT_DIR / "aso_spatial_metrics_3panel_audit.csv"
)
metrics_all.to_csv(figure_audit_path, index=False)

print("Wrote elevation-band metrics:", elevation_metrics_path)
print("Wrote 3x1 figure audit table:", figure_audit_path)
print(
    "Each elevation-band row includes Pearson r, MB, RMB (%), "
    "RMSE, RRMSE (%), and n for iSnobal and Noah-MP."
)


## Bulk snow-density comparison on the common Noah-MP grid

The density analysis uses **ASO snow depth > 100 mm as the reference screening
criterion**. It does not require iSnobal or Noah-MP snow depth to exceed 100 mm,
because doing so would allow the evaluated models to determine which cells enter
the validation sample.

All three sources must nevertheless have finite SWE and snow depth on a cell, and
model snow depth must be positive so that density is defined. The exact same cells
are then used for ASO, iSnobal, and Noah-MP density statistics.

For the compact publication-style output, use the rows with
`elevation_band == "All"`.


In [ ]:
# ============================================================
# BULK SNOW DENSITY: ASO SD > 100 mm, COMMON THREE-WAY SUPPORT
# ============================================================

DENSITY_ASO_SD_THRESHOLD_MM = 100.0


def bulk_density_kg_m3(swe_mm, snow_depth_mm):
    """Cell-wise bulk density from common-grid SWE and snow depth."""
    swe = np.asarray(swe_mm, dtype=float)
    depth = np.asarray(snow_depth_mm, dtype=float)

    out = np.full(swe.shape, np.nan, dtype=float)
    valid = (
        np.isfinite(swe)
        & np.isfinite(depth)
        & (depth > 0.0)
    )
    out[valid] = 1000.0 * swe[valid] / depth[valid]
    return out


def rmse(reference, model):
    reference = np.asarray(reference, dtype=float)
    model = np.asarray(model, dtype=float)
    valid = np.isfinite(reference) & np.isfinite(model)
    if not np.any(valid):
        return np.nan
    error = model[valid] - reference[valid]
    return float(np.sqrt(np.mean(error ** 2)))


def build_bulk_density_metrics():
    swe_dates = set(
        table6_common.loc[table6_common["variable"].eq("SWE"), "date"]
        .dt.normalize()
        .tolist()
    )
    sd_dates = set(
        table6_common.loc[table6_common["variable"].eq("Snow depth"), "date"]
        .dt.normalize()
        .tolist()
    )

    if swe_dates != sd_dates:
        only_swe = sorted(swe_dates - sd_dates)
        only_sd = sorted(sd_dates - swe_dates)
        raise ValueError(
            "SWE and snow-depth ASO date sets do not match, so density cannot "
            "be formed silently. "
            f"SWE-only dates={only_swe}; snow-depth-only dates={only_sd}"
        )

    dates = sorted(swe_dates)
    rows = []

    watershed_mask = (
        np.isfinite(noah_weights)
        & (noah_weights > 0)
        & np.isfinite(noah_dem)
    )

    for date in dates:
        # Existing workflow common-grid products.
        aso_swe = read_raster(date, "SWE", "aso_on_noah")
        isnobal_swe = read_raster(date, "SWE", "isnobal_on_noah")
        noah_swe = read_raster(date, "SWE", "noah_native")

        aso_sd = read_raster(date, "Snow depth", "aso_on_noah")
        isnobal_sd = read_raster(date, "Snow depth", "isnobal_on_noah")
        noah_sd = read_raster(date, "Snow depth", "noah_native")

        # Calculate density only after SWE and snow depth are on the common grid.
        density_aso = bulk_density_kg_m3(aso_swe, aso_sd)
        density_i = bulk_density_kg_m3(isnobal_swe, isnobal_sd)
        density_n = bulk_density_kg_m3(noah_swe, noah_sd)

        water_year = date.year + 1 if date.month >= 10 else date.year

        for elevation_band in ELEVATION_BANDS:
            lower_m, upper_m = ELEVATION_BANDS[elevation_band]

            band_mask = (
                watershed_mask
                & elevation_band_mask(noah_dem, elevation_band)
            )

            # Reference-based screening: ASO SD > 100 mm.
            aso_threshold_mask = (
                band_mask
                & np.isfinite(aso_sd)
                & (aso_sd > DENSITY_ASO_SD_THRESHOLD_MM)
            )

            # Common three-way source support before calculating statistics.
            common_input_mask = (
                aso_threshold_mask
                & np.isfinite(aso_swe)
                & np.isfinite(isnobal_swe)
                & np.isfinite(noah_swe)
                & np.isfinite(isnobal_sd)
                & np.isfinite(noah_sd)
                # Positive model depth is not an additional 100-mm screening
                # rule; it is required only because SWE / depth is undefined
                # at nonpositive depth.
                & (isnobal_sd > 0.0)
                & (noah_sd > 0.0)
            )

            common_density_mask = (
                common_input_mask
                & np.isfinite(density_aso)
                & np.isfinite(density_i)
                & np.isfinite(density_n)
            )

            n = int(np.count_nonzero(common_density_mask))

            if n == 0:
                aso_mean = np.nan
                i_mean = np.nan
                n_mean = np.nan
                r_i = np.nan
                r_n = np.nan
                mb_i = np.nan
                mb_n = np.nan
                rmse_i = np.nan
                rmse_n = np.nan
            else:
                a = density_aso[common_density_mask]
                i = density_i[common_density_mask]
                nmod = density_n[common_density_mask]

                aso_mean = float(np.mean(a))
                i_mean = float(np.mean(i))
                n_mean = float(np.mean(nmod))

                stats_i = pearson_mean_bias(a, i)
                stats_n = pearson_mean_bias(a, nmod)

                # Both comparisons must use the exact same common support.
                if int(stats_i["n"]) != n or int(stats_n["n"]) != n:
                    raise ValueError(
                        f"Density support mismatch for {date.date()} / "
                        f"{elevation_band}: common={n}, "
                        f"iSnobal={stats_i['n']}, Noah-MP={stats_n['n']}."
                    )

                r_i = float(stats_i["pearson_r"])
                r_n = float(stats_n["pearson_r"])
                mb_i = float(stats_i["mean_bias"])
                mb_n = float(stats_n["mean_bias"])
                rmse_i = rmse(a, i)
                rmse_n = rmse(a, nmod)

            rows.append({
                "date": pd.Timestamp(date),
                "water_year": int(water_year),
                "elevation_band": elevation_band,
                "elevation_range": elevation_range_text(elevation_band),
                "elevation_lower_m": (
                    np.nan if lower_m is None else float(lower_m)
                ),
                "elevation_upper_m": (
                    np.nan if upper_m is None else float(upper_m)
                ),
                "aso_sd_threshold_operator": ">",
                "aso_sd_threshold_mm": float(DENSITY_ASO_SD_THRESHOLD_MM),
                "n_aso_sd_gt_100_before_common_finite": int(
                    np.count_nonzero(aso_threshold_mask)
                ),
                "n_common_density": n,
                "aso_mean_density_kg_m3": aso_mean,
                "isnobal_mean_density_kg_m3": i_mean,
                "noahmp_mean_density_kg_m3": n_mean,
                "r_isnobal": r_i,
                "mb_isnobal_kg_m3": mb_i,
                "rmb_isnobal_pct": relative_metric_percent(mb_i, aso_mean),
                "rmse_isnobal_kg_m3": rmse_i,
                "rrmse_isnobal_pct": relative_metric_percent(rmse_i, aso_mean),
                "r_noahmp": r_n,
                "mb_noahmp_kg_m3": mb_n,
                "rmb_noahmp_pct": relative_metric_percent(mb_n, aso_mean),
                "rmse_noahmp_kg_m3": rmse_n,
                "rrmse_noahmp_pct": relative_metric_percent(rmse_n, aso_mean),
            })

    band_order = {band: i for i, band in enumerate(ELEVATION_BANDS)}
    out = pd.DataFrame(rows)
    out["_band_order"] = out["elevation_band"].map(band_order)
    out = (
        out.sort_values(["date", "_band_order"])
        .drop(columns="_band_order")
        .reset_index(drop=True)
    )
    return out


density_metrics = build_bulk_density_metrics()
density_metrics_all = density_metrics[
    density_metrics["elevation_band"].eq("All")
].copy()

density_by_elevation_path = (
    TABLE_OUTPUT_DIR / "aso_bulk_density_metrics_by_elevation.csv"
)
density_all_path = (
    TABLE_OUTPUT_DIR / "aso_bulk_density_metrics_all.csv"
)

density_metrics.to_csv(density_by_elevation_path, index=False)
density_metrics_all.to_csv(density_all_path, index=False)

print("Wrote density metrics by elevation:", density_by_elevation_path)
print("Wrote compact All-band density metrics:", density_all_path)
print(
    "Density method: common Noah grid -> ASO SD > 100 mm -> common finite "
    "three-way support -> cell-wise density -> equal-weight spatial statistics."
)

density_review_columns = [
    "date",
    "elevation_band",
    "n_common_density",
    "aso_mean_density_kg_m3",
    "isnobal_mean_density_kg_m3",
    "noahmp_mean_density_kg_m3",
    "r_isnobal",
    "mb_isnobal_kg_m3",
    "rmb_isnobal_pct",
    "rmse_isnobal_kg_m3",
    "rrmse_isnobal_pct",
    "r_noahmp",
    "mb_noahmp_kg_m3",
    "rmb_noahmp_pct",
    "rmse_noahmp_kg_m3",
    "rrmse_noahmp_pct",
]

density_metrics_all[density_review_columns]


In [ ]:

# ============================================================
# GROUPED ASO-DATE X AXIS
# ============================================================

def add_grouped_x_positions(frame, gap=WY_GAP):
    work = frame.sort_values("date").copy()
    groups = []
    cursor = 0.0

    for wy, sub in work.groupby("water_year", sort=True):
        sub = sub.sort_values("date")
        xs = np.arange(len(sub), dtype=float) + cursor

        for idx, x in zip(sub.index, xs):
            work.loc[idx, "_x"] = x

        groups.append({
            "water_year": int(wy),
            "start": float(xs[0]),
            "end": float(xs[-1]),
            "center": float((xs[0] + xs[-1]) / 2.0),
        })

        cursor = float(xs[-1] + 1.0 + gap)

    return work.sort_values("_x").reset_index(drop=True), groups


def plot_within_water_year(ax, frame, y_col, color, linewidth):
    # Same circular marker for all series; connect only within a water year.
    for _, sub in frame.groupby("water_year", sort=True):
        sub = sub.sort_values("_x")
        ax.plot(
            sub["_x"],
            sub[y_col],
            color=color,
            linewidth=linewidth,
            marker="o",
            markersize=MARKER_SIZE,
            markerfacecolor=color,
            markeredgecolor=color,
            zorder=3,
        )


def add_wy_separators(ax, groups):
    for left, right in zip(groups[:-1], groups[1:]):
        separator = (left["end"] + right["start"]) / 2.0
        ax.axvline(
            separator,
            color="0.78",
            linestyle="--",
            linewidth=1.0,
            zorder=0,
        )


def clean_left_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.0)
    ax.spines["bottom"].set_linewidth(1.0)
    ax.tick_params(
        axis="both",
        which="major",
        width=1.0,
        length=5,
    )


In [ ]:

# ============================================================
# 3×1 FIGURE
# ============================================================

def plot_spatial_metrics(
    metrics_table,
    variable,
    variable_display,
    output_stem,
    elevation_band="All",
):
    sub = metrics_table[
        metrics_table["variable"].eq(variable)
        & metrics_table["elevation_band"].eq(elevation_band)
    ].copy()

    if sub.empty:
        raise ValueError(
            f"No rows found for variable={variable!r}, "
            f"elevation_band={elevation_band!r}. "
            f"Available variables: "
            f"{sorted(metrics_table['variable'].unique().tolist())}; "
            f"available bands: "
            f"{sorted(metrics_table['elevation_band'].unique().tolist())}"
        )

    sub, wy_groups = add_grouped_x_positions(sub)

    fig, axes = plt.subplots(
        3,
        1,
        figsize=FIGSIZE,
        sharex=True,
        gridspec_kw={
            "height_ratios": [1.08, 1.0, 1.0],
            "hspace": 0.30,
        },
    )

    ax_a, ax_b, ax_c = axes

    # --------------------------------------------------------
    # A. ASO magnitude + model MB
    # --------------------------------------------------------
    ax_a_r = ax_a.twinx()

    plot_within_water_year(
        ax_a,
        sub,
        "aso_band_weighted_mean_mm",
        ASO_COLOR,
        ASO_LINEWIDTH,
    )
    plot_within_water_year(
        ax_a_r,
        sub,
        "mb_isnobal_mm",
        MODEL_COLORS["iSnobal"],
        MODEL_LINEWIDTH,
    )
    plot_within_water_year(
        ax_a_r,
        sub,
        "mb_noahmp_mm",
        MODEL_COLORS["Noah-MP"],
        MODEL_LINEWIDTH,
    )

    ax_a_r.axhline(
        0.0,
        color="0.45",
        linestyle="--",
        linewidth=1.0,
        zorder=1,
    )

    ax_a.set_ylabel(f"{variable_display} (mm)")
    ax_a_r.set_ylabel("MB (mm)")

    ax_a.set_title(
        "ASO magnitude and model MB",
        fontsize=PANEL_TITLE_SIZE,
        fontweight="normal",
        loc="center",
        pad=8,
    )

    aso_values = sub["aso_band_weighted_mean_mm"].to_numpy(dtype=float)
    aso_values = aso_values[np.isfinite(aso_values)]
    aso_max = float(np.nanmax(aso_values)) if aso_values.size else 1.0
    aso_top = max(
        50.0,
        float(np.ceil(aso_max / 50.0) * 50.0),
    )
    ax_a.set_ylim(0, aso_top+100)

    bias_values = np.abs(
        np.r_[
            sub["mb_isnobal_mm"].to_numpy(dtype=float),
            sub["mb_noahmp_mm"].to_numpy(dtype=float),
        ]
    )
    bias_values = bias_values[np.isfinite(bias_values)]
    bias_lim = (
        max(
            25.0,
            float(np.ceil(np.nanmax(bias_values) / 25.0) * 25.0),
        )
        if bias_values.size
        else 100.0
    )
    ax_a_r.set_ylim(-bias_lim, bias_lim)

    # One legend only for all three panels.
    handles = [
        Line2D(
            [], [],
            color=ASO_COLOR,
            linewidth=ASO_LINEWIDTH,
            marker="o",
            markersize=MARKER_SIZE,
            label=f"ASO",
        ),
        Line2D(
            [], [],
            color=MODEL_COLORS["iSnobal"],
            linewidth=MODEL_LINEWIDTH,
            marker="o",
            markersize=MARKER_SIZE,
            label="iSnobal",
        ),
        Line2D(
            [], [],
            color=MODEL_COLORS["Noah-MP"],
            linewidth=MODEL_LINEWIDTH,
            marker="o",
            markersize=MARKER_SIZE,
            label="Noah-MP",
        ),
    ]

    ax_a.legend(
        handles=handles,
        loc="upper right",
        ncol=1,
        frameon=True,
        facecolor="white",
        edgecolor="0.4",
        framealpha=0.9,
        fancybox=False,
        borderpad=0.5,
        handlelength=2.2,
    )

    # --------------------------------------------------------
    # B. Spatial RMSE
    # --------------------------------------------------------
    plot_within_water_year(
        ax_b,
        sub,
        "rmse_isnobal_mm",
        MODEL_COLORS["iSnobal"],
        MODEL_LINEWIDTH,
    )
    plot_within_water_year(
        ax_b,
        sub,
        "rmse_noahmp_mm",
        MODEL_COLORS["Noah-MP"],
        MODEL_LINEWIDTH,
    )

    ax_b.set_ylabel("(mm)")
    ax_b.set_title(
        "RMSE",
        fontsize=PANEL_TITLE_SIZE,
        fontweight="normal",
        loc="center",
        pad=8,
    )

    rmse_values = np.r_[
        sub["rmse_isnobal_mm"].to_numpy(dtype=float),
        sub["rmse_noahmp_mm"].to_numpy(dtype=float),
    ]
    rmse_values = rmse_values[np.isfinite(rmse_values)]
    rmse_top = (
        max(
            50.0,
            float(np.ceil(np.nanmax(rmse_values) / 50.0) * 50.0),
        )
        if rmse_values.size
        else 100.0
    )
    ax_b.set_ylim(0, rmse_top)

    # --------------------------------------------------------
    # C. Spatial Pearson r
    # --------------------------------------------------------
    plot_within_water_year(
        ax_c,
        sub,
        "r_isnobal",
        MODEL_COLORS["iSnobal"],
        MODEL_LINEWIDTH,
    )
    plot_within_water_year(
        ax_c,
        sub,
        "r_noahmp",
        MODEL_COLORS["Noah-MP"],
        MODEL_LINEWIDTH,
    )

    ax_c.set_ylabel("")
    ax_c.set_title(
        "Pearson correlation",
        fontsize=PANEL_TITLE_SIZE,
        fontweight="normal",
        loc="center",
        pad=8,
    )
    ax_c.set_ylim(0.6, 1.0)

    # --------------------------------------------------------
    # Shared x-axis
    # --------------------------------------------------------
    for ax in axes:
        add_wy_separators(ax, wy_groups)
        clean_left_axis(ax)

    # Retain right spine only for Panel A's dual y-axis.
    ax_a_r.spines["top"].set_visible(False)
    ax_a_r.spines["left"].set_visible(False)
    ax_a_r.spines["right"].set_linewidth(1.0)
    ax_a_r.tick_params(
        axis="y",
        which="major",
        width=1.0,
        length=5,
    )

    # Date labels appear only once, at the bottom.
    ax_c.set_xticks(sub["_x"])
    ax_c.set_xticklabels(
        [d.strftime("%b %d") for d in sub["date"]],
        rotation=0,
        ha="center",
        fontsize=12,
    )

    # Second x-label row: centered water-year labels.
    transform = blended_transform_factory(
        ax_c.transData,
        ax_c.transAxes,
    )
    for group in wy_groups:
        ax_c.text(
            group["center"],
            -0.21,
            f"WY{group['water_year']}",
            transform=transform,
            ha="center",
            va="top",
            fontsize=BASE_FONT_SIZE,
            fontweight="normal",
        )

    ax_a.tick_params(axis="x", labelbottom=False)
    ax_b.tick_params(axis="x", labelbottom=False)

    xmin = float(sub["_x"].min() - 0.45)
    xmax = float(sub["_x"].max() + 0.45)
    for ax in axes:
        ax.set_xlim(xmin, xmax)

    # No overall figure title.
    fig.subplots_adjust(
        top=0.97,
        bottom=0.12,
        left=0.11,
        right=0.88,
        hspace=0.30,
    )

    # raw-publication-panel: data axes only; the twin axis is not a panel.
    for index, panel in enumerate(axes):
        if not any(t.get_gid() == 'publication-panel-label' for t in panel.texts):
            label = panel.annotate(f'({chr(97 + index)})', xy=(0, 1), xycoords='axes fraction',
                xytext=(0, 8), textcoords='offset points', ha='left', va='bottom',
                fontsize=18, fontfamily='Nimbus Sans', fontweight='normal', color='black',
                annotation_clip=False, clip_on=False)
            label.set_gid('publication-panel-label')

    png_path = FIGURE_OUTPUT_DIR / f"{output_stem}.png"
    pdf_path = FIGURE_OUTPUT_DIR / f"{output_stem}.pdf"

    fig.savefig(
        png_path,
        dpi=SAVE_DPI,
        bbox_inches="tight",
        pad_inches=0.05,
    )
    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        pad_inches=0.05,
    )

    print("Saved PNG:", png_path)
    print("Saved PDF:", pdf_path)

    return fig, axes, sub


In [ ]:

# ============================================================
# SWE FIGURE
# ============================================================

fig_swe, axes_swe, swe_metrics = plot_spatial_metrics(
    metrics_table=metrics,
    variable="SWE",
    variable_display="SWE",
    output_stem="aso_spatial_metrics_swe_3panel",
)

plt.show()


In [ ]:

# ============================================================
# SNOW-DEPTH FIGURE
# ============================================================

fig_sd, axes_sd, sd_metrics = plot_spatial_metrics(
    metrics_table=metrics,
    variable="Snow depth",
    variable_display="Snow depth",
    output_stem="aso_spatial_metrics_snow_depth_3panel",
)

plt.show()


## Notes

- Elevation bands use the exact workflow definitions:
  - Lower: `< 2896 m`
  - Middle: `>= 2896 m and < 3353 m`
  - Upper: `>= 3353 m`
  - All: all valid watershed cells.
- `r` is the **Pearson correlation coefficient**, not \(R^2\).
- SWE/snow-depth `MB` and `RMSE` come from Table 6 for each date, variable,
  product, and elevation band.
- `RMB (%) = 100 x MB / mean(ASO)` and preserves the **model - ASO** sign.
- `RRMSE (%) = 100 x RMSE / mean(ASO)` and is nonnegative.
- The main SWE/snow-depth CSV containing **Lower, Middle, Upper, and All** is
  written to `output/output1/tables/aso_spatial_metrics_by_elevation.csv`.
- The existing 3x1 SWE and snow-depth figures remain whole-watershed (`All`)
  figures.

### Bulk-density method and outputs

- Density is calculated **after** ASO and iSnobal SWE/snow depth have already
  been area-averaged to the common Noah-MP grid.
- Screening uses **ASO snow depth > 100 mm**.
- iSnobal and Noah-MP are **not** required to exceed 100 mm snow depth.
- All three sources must have finite SWE and snow depth on the same common-grid
  cell; model snow depth must be positive because density is undefined at
  nonpositive depth.
- Density is calculated cell by cell as
  `1000 * SWE_mm / snow_depth_mm` in `kg m^-3`.
- The exact same retained cells are used for both model comparisons.
- Each retained Noah-grid cell receives equal statistical weight.
- No upper-density clipping is applied.
- `aso_mean_density_kg_m3` is therefore a single shared ASO reference mean for
  both model comparisons on each date/elevation-band row.
- Full density metrics are written to
  `output/output1/tables/aso_bulk_density_metrics_by_elevation.csv`.
- A compact whole-watershed (`All`) density table is written to
  `output/output1/tables/aso_bulk_density_metrics_all.csv`.
- The notebook itself applies no additional ASO QC; density inherits the ASO
  version used when the workflow's common-grid rasters were generated.
